# 第 6 周练习：「价格合适」顶点项目（Price is Right）

## 练习目标

本笔记本把第 6 周 5 天内容串成一条**价格预测**流水线：根据亚马逊商品描述估计售价。

| 天数 | 你会练到的概念 |
|------|----------------|
| 第 1 天 | 数据管理：加载、探索价格/文本长度/类别分布 |
| 第 2 天 | 预处理：用 LLM 把描述重写成标准 `summary` |
| 第 3 天 | 基线与传统 ML：随机、常量、线性回归、词袋、随机森林、XGBoost |
| 第 4 天 | 深度学习与 LLM：神经网络、零样本（zero-shot）推理（`gpt-4.1-nano`） |
| 第 5 天 | 微调（fine-tuning）：在价格任务上微调 GPT-4.1-nano |

## 怎么跑

1. 从仓库根目录或 `week6` 目录运行（下方单元格会自动定位路径）
2. `.env` 里准备 `HF_TOKEN`（HuggingFace）；若跑 LLM/微调再加 `OPENAI_API_KEY`
3. 免费快速试跑请保持 `LITE_MODE = True`（约 20k 条、Hub 上已预处理的数据）


## 设置（Setup）

先把 `week6` 加进 `sys.path`，并登录 HuggingFace，后面才能 `import pricer` 与拉数据集。


In [ ]:
# ========== 定位 week6 目录：让 pricer 包可以被 import ==========

# 导入 sys：改 Python 模块搜索路径
import sys
# 导入 os：切换工作目录、读环境变量
import os
# 从 pathlib 导入 Path：用面向对象方式拼路径，比字符串拼接更稳
from pathlib import Path

# 当前工作目录（Current Working Directory）
cwd = Path.cwd()
# 依次在 cwd、父目录（仓库根）、再上一级（社区贡献子目录）里找 week6
for candidate in [cwd, cwd.parent, cwd.parent.parent]:
    week6_dir = candidate / "week6"
    # 找到存在的 week6 就停
    if week6_dir.exists():
        break
else:
    # 都找不到时退回 cwd，避免后面路径完全崩掉
    week6_dir = cwd
# 把 week6 插到 sys.path 最前，保证优先 import 到课程里的 pricer
sys.path.insert(0, str(week6_dir))
# 切到 week6，相对路径（如 jsonl/）才和官方笔记本一致
os.chdir(week6_dir)
# 打印确认当前工作目录
print(f"Working directory: {os.getcwd()}")


In [ ]:
# ========== 环境变量 + HuggingFace 登录 ==========

# 从 dotenv 导入 load_dotenv：把 .env 密钥读进环境变量（Environment Variables）
from dotenv import load_dotenv
# 从 huggingface_hub 导入 login：用 token 登录 Hub，才能拉私有/限流数据集
from huggingface_hub import login

# override=True：.env 覆盖已有同名环境变量
load_dotenv(override=True)
# 读取 HF_TOKEN；没有也不抛错（用 get）
hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    # 有 token：登录；add_to_git_credential=True 顺带写 git 凭证
    login(hf_token, add_to_git_credential=True)
    print("HuggingFace login successful")
else:
    # 无 token：提示去 .env 配置（文案保持英文，和原逻辑一致）
    print("HF_TOKEN not set — add to .env to load datasets from Hub")


## 第 1 天：数据管理

加载原始商品数据，探索价格、文本长度、类别分布；精选数据来自 HuggingFace Hub。

原始语料参考：[McAuley-Lab/Amazon-Reviews-2023](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023)。


In [ ]:
# ========== 从 Hub 加载原始 Item（含 full 全文）做探索 ==========

# LITE_MODE=True：用 20k 轻量集，免费、跑得快；False 才上完整集
LITE_MODE = True  # Use lite dataset (20k train) for fast, free runs
# Hub 上数据集所属用户名（字符串必须保持原样）
USERNAME = "ed-donner"

# 从课程 pricer 包导入 Item：统一的商品数据结构 + from_hub 加载器
from pricer.items import Item
# 按 LITE_MODE 选择 raw lite / raw full 数据集名
raw_dataset = f"{USERNAME}/items_raw_lite" if LITE_MODE else f"{USERNAME}/items_raw_full"
# 一次取出 train / val / test 三个划分
train_raw, val_raw, test_raw = Item.from_hub(raw_dataset)
# 拼成一个大列表，方便画全局分布图
items_raw = train_raw + val_raw + test_raw
# 打印加载条数（千分位逗号）
print(f"Loaded {len(items_raw):,} raw items")


In [ ]:
# ========== 第 1 天：价格分布 vs 文本长度分布 ==========

# 导入 matplotlib.pyplot：画直方图
import matplotlib.pyplot as plt
# 从 collections 导入 Counter：后面统计类别频次
from collections import Counter

# 抽出每条商品的价格
prices = [item.price for item in items_raw]
# 抽出 full 文本长度；None 当空串，避免 len(None) 报错
lengths = [len(item.full or "") for item in items_raw]

# 画布：宽 12、高 5，左右两个子图
plt.figure(figsize=(12, 5))
# 左图：价格直方图
plt.subplot(1, 2, 1)
plt.hist(prices, bins=50, color="blueviolet", rwidth=0.9)
plt.xlabel("Price ($)")
plt.ylabel("Count")
# 标题里带上平均价，一眼看中心趋势
plt.title(f"Price distribution (avg ${sum(prices)/len(prices):.1f})")
# 右图：文本长度直方图
plt.subplot(1, 2, 2)
plt.hist(lengths, bins=50, color="skyblue", rwidth=0.9)
plt.xlabel("Text length (chars)")
plt.ylabel("Count")
plt.title(f"Text length (avg {sum(lengths)/len(lengths):.0f})")
# 自动紧凑布局，避免标题/轴标签重叠
plt.tight_layout()
plt.show()


In [ ]:
# ========== 按 category 统计条数并画柱状图 ==========

# Counter：类别 -> 出现次数
cat_counts = Counter([item.category for item in items_raw])
# 新画布
plt.figure(figsize=(10, 4))
# 横轴类别名，纵轴计数
plt.bar(cat_counts.keys(), cat_counts.values(), color="goldenrod")
# 类别名较长：旋转 30° 并对齐右端，避免挤在一起
plt.xticks(rotation=30, ha="right")
plt.title("Items per category")
plt.show()


## 第 2 天：数据预处理

用 LLM 把杂乱商品描述重写成统一格式的 `summary`，方便下游模型吃同一套输入。

本练习直接加载已经带好 `summary` 的预处理数据集（`items_lite` / `items_full`），不必现场再调一次重写 API。


In [ ]:
# ========== 预处理用的 system prompt（第 2 天生成 summary 时用） ==========
# 注意：下方三引号字符串是会影响模型行为的 prompt，必须保持英文原文，不要翻译

SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""


In [ ]:
# ========== 加载带 summary 的预处理数据，供建模使用 ==========

# lite / full 二选一，与前面的 LITE_MODE 对齐
dataset = f"{USERNAME}/items_lite" if LITE_MODE else f"{USERNAME}/items_full"
# Item.from_hub 返回 train / val / test
train, val, test = Item.from_hub(dataset)
# 打印三个划分的规模
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")


In [ ]:
# ========== 看一条预处理后的 summary 长什么样 ==========

print("Sample item summary (pre-processed):")
# train[0].summary：标准化短描述，后面词袋/神经网络都吃这个字段
print(train[0].summary)


## 第 3 天：基线与传统机器学习

用 `pricer.evaluator` 的 `evaluate` 统一评估：

- 简单基线：随机猜价、训练集均价常量
- 传统 ML：手工特征线性回归、词袋（Bag-of-Words）+ 线性/随机森林/XGBoost


In [ ]:
# ========== 第 3 天所需依赖：随机数、数值、表格、sklearn、evaluate ==========

# 导入 random：随机基线用
import random
# 导入 numpy：数组与随机种子
import numpy as np
# 导入 pandas：把特征字典收成 DataFrame
import pandas as pd
# 线性回归：最简单的监督学习回归器
from sklearn.linear_model import LinearRegression
# CountVectorizer：词袋特征（词频向量）
from sklearn.feature_extraction.text import CountVectorizer
# RandomForestRegressor：树集成回归
from sklearn.ensemble import RandomForestRegressor
# evaluate：课程统一的定价器评测入口（传入可调用对象 + test 集）
from pricer.evaluator import evaluate


In [ ]:
# ========== 基线 1：完全随机猜一个 1~999 的整数价 ==========

# 固定种子，复现实验
random.seed(42)

# 定价器：签名是 item -> 预测价格；这里故意忽略 item 内容
def random_pricer(item):
    return random.randrange(1, 1000)

# 在 test 上抽 100 条评估，建立「最差也该超过它」的参照
evaluate(random_pricer, test, size=100)


In [ ]:
# ========== 基线 2：恒定预测 = 训练集平均价 ==========

# 训练集价格均值：任何模型至少应打赢「永远报同一个数」
train_avg = sum(item.price for item in train) / len(train)

# 常量定价器：忽略 item，总是返回 train_avg
def constant_pricer(item):
    return train_avg

evaluate(constant_pricer, test, size=100)


In [ ]:
# ========== 手工特征：重量、重量是否缺失、summary 文本长度 ==========

# 从单个 Item 抽出数值特征字典
def get_features(item):
    return {
        # 重量缺失时用 0 占位
        "weight": item.weight or 0,
        # 显式标记「重量未知」，避免模型把 0 当成真的很轻
        "weight_unknown": 1 if (item.weight or 0) == 0 else 0,
        # summary 字符数：描述长短有时和品类/价格相关
        "text_length": len(item.summary or ""),
    }

# 一批 Item -> 带 price 列的 DataFrame
def list_to_df(items):
    # 逐条提特征
    fs = [get_features(i) for i in items]
    df = pd.DataFrame(fs)
    # 追加监督标签：真实价格
    df["price"] = [i.price for i in items]
    return df

# 训练集 / 测试集表格
train_df = list_to_df(train)
test_df = list_to_df(test)
# 参与拟合的特征列名（顺序固定，预测时也要同一顺序）
feature_cols = ["weight", "weight_unknown", "text_length"]

# 拆出 X（特征）与 y（价格）
X_train, y_train = train_df[feature_cols], train_df["price"]
X_test, y_test = test_df[feature_cols], test_df["price"]


In [ ]:
# ========== 手工特征上的线性回归定价器 ==========

# 固定 numpy 随机性，便于复现
np.random.seed(42)
# 创建线性回归模型
lr_model = LinearRegression()
# 在训练特征上拟合
lr_model.fit(X_train, y_train)

# 推理：先提特征 -> DataFrame 对齐列 -> predict -> 价格下限截到 0
def linear_pricer(item):
    f = get_features(item)
    pred = lr_model.predict(pd.DataFrame([f])[feature_cols])[0]
    return max(0, pred)

evaluate(linear_pricer, test, size=100)


In [ ]:
# ========== 词袋（Bag-of-Words）+ 线性回归 ==========

# 语料：每条训练样本的 summary 文本
documents = [item.summary for item in train]
# 标签：浮点价格数组
prices_arr = np.array([float(item.price) for item in train], dtype=float)

np.random.seed(42)
# 最多 2000 维词袋，去掉英文停用词
vectorizer = CountVectorizer(max_features=2000, stop_words="english")
# fit_transform：学词表并得到稀疏矩阵 X_vec
X_vec = vectorizer.fit_transform(documents)

# 在词袋特征上再训一个线性回归
bow_model = LinearRegression()
bow_model.fit(X_vec, prices_arr)

# 预测：transform 单条 summary（不要再 fit，否则词表漂移）
def bow_linear_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(0, bow_model.predict(x)[0])

evaluate(bow_linear_pricer, test, size=100)


In [ ]:
# ========== 随机森林：在词袋特征上拟合（lite 时截前 15k 加速） ==========

# 训练子集大小：最多 15000，避免 RF 在全量上太慢
subset = min(15_000, len(train))
# 100 棵树；n_jobs=4 并行拟合
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
# 用与线性词袋相同的 X_vec / prices_arr，但只取前 subset 行
rf_model.fit(X_vec[:subset], prices_arr[:subset])

def rf_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

evaluate(rf_pricer, test, size=100)


In [ ]:
# ========== XGBoost（可选：没装包就跳过，不中断整本笔记本） ==========

try:
    # 延迟导入：未安装时走 except
    import xgboost as xgb
    # 梯度提升回归：200 棵树、学习率 0.1
    xgb_model = xgb.XGBRegressor(n_estimators=200, random_state=42, n_jobs=4, learning_rate=0.1)
    xgb_model.fit(X_vec, prices_arr)

    def xgb_pricer(item):
        x = vectorizer.transform([item.summary])
        return max(0, xgb_model.predict(x)[0])

    evaluate(xgb_pricer, test, size=100)
except ImportError:
    # 提示安装命令的英文文案保持原样（依赖程序/用户判断）
    print("XGBoost not installed — skip with: pip install xgboost")


## 第 4 天：深度学习与大语言模型（LLM）

1. 在哈希词袋特征上训练一个普通前馈神经网络（Neural Network）
2. 再用前沿 LLM 做零样本价格估计（需要 API Key）


In [ ]:
# ========== 深度学习相关导入 ==========

# PyTorch 主包
import torch
# nn：网络层与模块基类
import torch.nn as nn
# optim：优化器（这里用 Adam）
import torch.optim as optim
# DataLoader / TensorDataset：小批量训练循环
from torch.utils.data import DataLoader, TensorDataset
# HashingVectorizer：固定维度哈希词袋，不必存巨大词表
from sklearn.feature_extraction.text import HashingVectorizer
# train_test_split：再划一小块验证集（本练习主要用于切 tensor）
from sklearn.model_selection import train_test_split
# 笔记本友好进度条
from tqdm.notebook import tqdm


In [ ]:
# ========== 用 HashingVectorizer 得到二值词袋，并打成 Tensor ==========

np.random.seed(42)
# n_features=5000：哈希到 5000 维；binary=True：只记出现与否
hv = HashingVectorizer(n_features=5000, stop_words="english", binary=True)
# 复用前面的 documents（训练集 summary 列表）
X_hv = hv.fit_transform(documents)
# 价格标签
y_hv = np.array([float(i.price) for i in train], dtype=float)

# 稀疏矩阵 -> 稠密 FloatTensor，供 nn.Linear 使用
X_t = torch.FloatTensor(X_hv.toarray())
# 标签加一维变成 (N,1)，与网络输出形状对齐
y_t = torch.FloatTensor(y_hv).unsqueeze(1)
# 切出很小的验证比例（0.01）；本格主要用训练集 loader
X_tr, X_vl, y_tr, y_vl = train_test_split(X_t, y_t, test_size=0.01, random_state=42)
# batch_size=64，打乱顺序做 SGD 风格更新
loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)


In [ ]:
# ========== 定义多层感知机并训练 2 个 epoch ==========

# 前馈网络：输入维 -> 128 -> 64 -> 64 -> 64 -> 1（回归一个价格）
class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        # Sequential 把线性层与 ReLU 串起来
        self.layers = nn.Sequential(
            nn.Linear(input_size, 128), nn.ReLU(),
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, 1),
        )

    # 前向：把输入送进 self.layers
    def forward(self, x):
        return self.layers(x)

# 按特征维度实例化
nn_model = NeuralNetwork(X_t.shape[1])
# 均方误差：回归常用损失
loss_fn = nn.MSELoss()
# Adam 优化器，学习率 0.001
opt = optim.Adam(nn_model.parameters(), lr=0.001)

# 只训 2 轮：演示流程；要提分可加大 epoch
for epoch in range(2):
    nn_model.train()
    for bx, by in tqdm(loader):
        # 清梯度，避免累加
        opt.zero_grad()
        # 前向 + 算损失
        loss = loss_fn(nn_model(bx), by)
        # 反向传播
        loss.backward()
        # 参数更新
        opt.step()


In [ ]:
# ========== 把训练好的网络包成 evaluate 可用的定价器 ==========

def nn_pricer(item):
    # eval 模式：关闭 dropout 等训练期行为（本网虽无 dropout，习惯上仍写）
    nn_model.eval()
    # no_grad：推理不算梯度，省内存
    with torch.no_grad():
        # 同一套 hv.transform，保证特征空间一致
        x = hv.transform([item.summary])
        x = torch.FloatTensor(x.toarray())
        # 取第 0 个样本的标量输出
        out = nn_model(x)[0].item()
    # 价格不能为负
    return max(0, out)

evaluate(nn_pricer, test, size=100)


In [ ]:
# ========== LLM 零样本估价（需要 OPENAI_API_KEY；默认注释掉以免扣费） ==========

# 构造 Chat messages：只含 user，要求模型只回价格
def messages_for(item):
    # prompt 字符串必须保持英文原文
    msg = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": msg}]

# 通过 litellm.completion 调 openai/gpt-4.1-nano
def gpt_nano_pricer(item):
    from litellm import completion
    r = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return r.choices[0].message.content

# 若要真实调用 API：取消下一行注释（会消耗额度）
# evaluate(gpt_nano_pricer, test, size=50)


## 第 5 天：微调前沿模型

在少量「商品 summary → 价格」样本上微调 GPT-4.1-nano，让模型更贴合本任务。

需要 `OPENAI_API_KEY`，且会产生 API 费用；下面先准备 JSONL，真正提交微调作业的代码保持注释。


In [ ]:
# ========== 准备微调数据：每行一条 messages JSON（JSONL） ==========

import json

# 单条样本：user 问价 + assistant 给带 $ 的真价（监督信号）
def ft_messages_for(item):
    # 训练用 prompt 保持英文原文
    msg = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [
        {"role": "user", "content": msg},
        {"role": "assistant", "content": f"${item.price:.2f}"},
    ]

# 多条 Item -> JSONL 文本（每行一个 {"messages": [...]}）
def make_jsonl(items):
    lines = []
    for item in items:
        msgs = ft_messages_for(item)
        lines.append(json.dumps({"messages": msgs}))
    return "\n".join(lines)

# 小子集即可演示流程（OpenAI 常建议先从几十到一百级试起）
ft_train = train[:100]
ft_val = val[:50]


In [ ]:
# ========== 把 JSONL 写到 week6/jsonl/ 目录 ==========

import os
# 目录不存在就创建
os.makedirs("jsonl", exist_ok=True)
# 训练集文件
with open("jsonl/fine_tune_train.jsonl", "w") as f:
    f.write(make_jsonl(ft_train))
# 验证集文件
with open("jsonl/fine_tune_val.jsonl", "w") as f:
    f.write(make_jsonl(ft_val))
print("JSONL files written")


In [ ]:
# ========== 上传文件并创建微调作业（默认全注释：避免误扣费） ==========
# 需要 OPENAI_API_KEY，且确认愿意产生费用后再取消注释

# from openai import OpenAI
# client = OpenAI()
# with open("jsonl/fine_tune_train.jsonl", "rb") as f:
#     train_file = client.files.create(file=f, purpose="fine-tune")
# with open("jsonl/fine_tune_val.jsonl", "rb") as f:
#     val_file = client.files.create(file=f, purpose="fine-tune")
# job = client.fine_tuning.jobs.create(
#     training_file=train_file.id,
#     validation_file=val_file.id,
#     model="gpt-4.1-nano-2025-04-14",
#     seed=42,
#     hyperparameters={"n_epochs": 1, "batch_size": 1},
#     suffix="pricer",
# )
# job_id = job.id
# print(f"Fine-tune job: {job_id}")


In [ ]:
# ========== 作业完成后：用微调模型估价并 evaluate（默认注释） ==========

# fine_tuned = client.fine_tuning.jobs.retrieve(job_id).fine_tuned_model
# def ft_pricer(item):
#     r = client.chat.completions.create(
#         model=fine_tuned,
#         messages=[{"role": "user", "content": f"Estimate the price...\n\n{item.summary}"}],
#         max_tokens=7,
#     )
#     return r.choices[0].message.content
# evaluate(ft_pricer, test)


## 总结

这条顶点流水线覆盖：

- **第 1 天**：亚马逊评论衍生的数据管理与分布探索
- **第 2 天**：基于 LLM 的标准化描述（`summary`）
- **第 3 天**：基线（随机、常量）与传统 ML（线性、RF、XGBoost）
- **第 4 天**：神经网络 + 零样本 LLM 推理
- **第 5 天**：微调 GPT-4.1-nano 做价格预测

想冲更高分：把 `LITE_MODE = False` 换成完整数据集。课程里用约 20k 样本微调时，平均误差可到约 $68 量级（视实验设定而变）。
